In [4]:
import re

class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

class ParserError(Exception):
    pass


class MiniCompiler:
    def __init__(self, source, env):
        # TUGAS 1: Menambahkan simbol '^'
        self._tokens = iter(re.findall(r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[+*/()\-^]', source) + ['?'])
        self._current = None
        self._env = env
        self._temp_count = 0
        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (expected == "ID" and self._current.isalnum()):
            raise ParserError(f"Expected {expected}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def factor(self):
        token = self._current

        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))

        elif token is not None and token.isidentifier():
            self.advance()

            if token not in self._env:
                raise Exception(f"Variabel '{token}' tidak ditemukan di symbol table")

            return Var(token)

        elif token == '(':
            self.advance()
            node = self.expr()
            self.expect(')')
            return node

        else:
            raise ParserError(f"Token tidak valid: {token}")

    # TUGAS 2: Tambahkan fungsi power()
    def power(self):
        node = self.factor()

        while self._current == '^':
            op = self._current
            self.advance()
            node = BinOp(node, op, self.factor())

        return node

    def term(self):
        # TUGAS 3: term() harus memanggil power()
        node = self.power()

        while self._current in ('*', '/'):
            op = self._current
            self.advance()
            node = BinOp(node, op, self.power())

        return node

    def expr(self):
        node = self.term()

        while self._current in ('+', '-'):
            op = self._current
            self.advance()
            node = BinOp(node, op, self.term())

        return node

    def new_temp(self):
        self._temp_count += 1
        return f"t{self._temp_count}"

    def generate_tac(self, node):
        if isinstance(node, Num):
            return str(node.value)

        elif isinstance(node, Var):
            return node.name

        elif isinstance(node, BinOp):
            left = self.generate_tac(node.left)
            right = self.generate_tac(node.right)

            temp = self.new_temp()
            print(f"{temp} = {left} {node.op} {right}")
            return temp


# Uji Coba
source_code = "a ^ 2 + b * c"
symbol_table = {'a': 5, 'b': 10, 'c': 2}

try:
    print(f"Input: {source_code}")

    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr()

    print("\n--- Output Three Address Code (TAC) ---")
    compiler.generate_tac(ast_root)

except Exception as e:
    print(f"Error: {e}")


Input: a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2


1. Mengapa fungsi power() harus dipanggil di dalam term(), bukan sebaliknya?

Karena operator pangkat (^) memiliki prioritas lebih tinggi dibandingkan perkalian (*) dan pembagian (/). Dengan memanggil power() di dalam term(), maka operasi pangkat akan diproses terlebih dahulu sebelum operasi kali atau bagi. Hal ini mengikuti konsep operator precedence dalam matematika.

2. Apa yang terjadi pada fase Analisis Semantik jika variabel z tidak ada di symbol_table?

Compiler akan menghasilkan error karena variabel z tidak didefinisikan di dalam symbol_table. Pada fase analisis semantik, compiler memeriksa apakah variabel yang digunakan sudah terdaftar atau belum.

3. Mengapa instruksi a ^ 2 harus muncul sebelum instruksi + pada TAC?

Karena operasi pangkat memiliki prioritas lebih tinggi daripada penjumlahan. TAC harus mengikuti urutan evaluasi operasi matematika sehingga hasil dari a ^ 2 harus dihitung terlebih dahulu sebelum digunakan pada operasi +.